# Template Manager Notebook

This notebook creates a directory listing of templates, saves it to a Markdown file, and uses it to update base.html and server.py.

## Import Required Libraries

In this section, we import the necessary Python libraries for working with the file system and performing directory operations.

In [ ]:
# Import Required Libraries
import os
import pathlib
from pathlib import Path
import re
import shutil
import datetime

## List Files in Directory

Now we'll use the pathlib library to list all files in the 'src/web/templates' directory and store the results in a structured format.

In [ ]:
# Define the template directory path
template_dir = Path("src/web/templates")

# Check if directory exists
if not template_dir.exists():
    print(f"Warning: Directory {template_dir} does not exist")
    # Create directory for demonstration if it doesn't exist
    template_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {template_dir}")

# Get all files in the templates directory
template_files = sorted(list(template_dir.glob("*.html")))

# Display the found template files
print(f"Found {len(template_files)} template files:")
for file in template_files:
    print(f"- {file.name}")

## Write Directory Listing to Markdown

We'll now create a structured Markdown file containing the directory listing with additional metadata about each template file.

In [ ]:
# Create a markdown file with the template listing
md_path = Path("docs/templates_listing.md")

# Make sure the directory exists
md_path.parent.mkdir(parents=True, exist_ok=True)

# Current timestamp for the markdown file
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(md_path, "w") as md_file:
    md_file.write(f"# Templates Directory Listing\n\n")
    md_file.write(f"*Generated on: {timestamp}*\n\n")
    md_file.write("## Available Templates\n\n")
    
    if not template_files:
        md_file.write("No template files found.\n")
    else:
        md_file.write("| File Name | Size (bytes) | Last Modified |\n")
        md_file.write("|-----------|--------------|---------------|\n")
        
        for file in template_files:
            size = file.stat().st_size
            last_modified = datetime.datetime.fromtimestamp(file.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            md_file.write(f"| {file.name} | {size} | {last_modified} |\n")
    
    md_file.write("\n## Template Structure\n\n")
    md_file.write("This section provides information about the relationship between templates.\n\n")

print(f"Created Markdown file: {md_path}")

## Analyze Template Relationships

Before updating files, let's analyze how templates are related to understand their inheritance structure.

In [ ]:
# Function to analyze template relationships
def analyze_template_relationships(template_files):
    relationships = {}
    content_map = {}
    
    # First, read all template files
    for file in template_files:
        with open(file, 'r', encoding='utf-8') as f:
            try:
                content = f.read()
                content_map[file.name] = content
            except UnicodeDecodeError:
                print(f"Warning: Could not read {file.name} as text. It may be a binary file.")
                content_map[file.name] = ""
    
    # Then analyze for extends and includes
    for filename, content in content_map.items():
        relationships[filename] = {
            "extends": [],
            "includes": [],
            "referenced_by": []
        }
        
        # Look for {% extends "template.html" %} pattern
        extends_matches = re.findall(r'{%\s*extends\s*[\'"]([^\'"]*)[\'"]\s*%}', content)
        for template in extends_matches:
            relationships[filename]["extends"].append(template)
            
        # Look for {% include "template.html" %} pattern
        include_matches = re.findall(r'{%\s*include\s*[\'"]([^\'"]*)[\'"]\s*%}', content)
        for template in include_matches:
            relationships[filename]["includes"].append(template)
    
    # Build referenced_by relationships
    for filename, rels in relationships.items():
        for extended in rels["extends"]:
            if extended in relationships:
                relationships[extended]["referenced_by"].append({"file": filename, "type": "extends"})
        for included in rels["includes"]:
            if included in relationships:
                relationships[included]["referenced_by"].append({"file": filename, "type": "includes"})
    
    return relationships

# Analyze template relationships
if template_files:
    relationships = analyze_template_relationships(template_files)
    
    # Update the markdown file with relationship information
    with open(md_path, "a") as md_file:
        md_file.write("### Template Inheritance\n\n")
        
        for filename, relation in relationships.items():
            md_file.write(f"#### {filename}\n\n")
            
            # Extensions
            if relation["extends"]:
                md_file.write("Extends:\n")
                for template in relation["extends"]:
                    md_file.write(f"- {template}\n")
            else:
                md_file.write("Extends: None\n")
            
            # Includes
            if relation["includes"]:
                md_file.write("\nIncludes:\n")
                for template in relation["includes"]:
                    md_file.write(f"- {template}\n")
            else:
                md_file.write("\nIncludes: None\n")
            
            # Referenced by
            if relation["referenced_by"]:
                md_file.write("\nReferenced by:\n")
                for ref in relation["referenced_by"]:
                    md_file.write(f"- {ref['file']} ({ref['type']})\n")
            else:
                md_file.write("\nReferenced by: None\n")
            
            md_file.write("\n")
    
    print("Template relationships analyzed and added to markdown file")
else:
    print("No template files to analyze relationships")

## Update base.html

Now we'll update the base.html file to ensure it correctly references all necessary templates and includes.

In [ ]:
# Path to base.html
base_html_path = template_dir / "base.html"

# Check if base.html exists
if not base_html_path.exists():
    print(f"Warning: {base_html_path} does not exist")
    # Create a basic base.html template for demonstration
    with open(base_html_path, "w") as base_file:
        base_file.write("""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Default Title{% endblock %}</title>
    <!-- CSS includes will be added here -->
</head>
<body>
    <!-- Navigation will be added here -->
    
    <div class="container">
        {% block content %}
        {% endblock %}
    </div>
    
    <!-- Footer will be added here -->
    
    <!-- JS includes will be added here -->
</body>
</html>
""")
        print(f"Created a basic {base_html_path} for demonstration")

# Read the current base.html file
with open(base_html_path, "r") as file:
    base_content = file.read()

# Create a backup of the original file
backup_path = base_html_path.with_suffix(".html.bak")
shutil.copy2(base_html_path, backup_path)
print(f"Created backup of base.html at {backup_path}")

# Update the base.html file to include references to other templates if needed
# For this example, we'll add a navigation include and footer include if they don't exist

# Check if we have a navigation template
nav_template = template_dir / "navigation.html"
if not nav_template.exists():
    # Create a simple navigation template
    with open(nav_template, "w") as nav_file:
        nav_file.write("""<!-- Navigation Template -->
<nav class="navbar navbar-expand-lg navbar-light bg-light">
    <div class="container">
        <a class="navbar-brand" href="/">ImpressionCore</a>
        <button class="navbar-toggler" type="button" data-toggle="collapse" data-target="#navbarNav">
            <span class="navbar-toggler-icon"></span>
        </button>
        <div class="collapse navbar-collapse" id="navbarNav">
            <ul class="navbar-nav ml-auto">
                <li class="nav-item"><a class="nav-link" href="/">Home</a></li>
                <li class="nav-item"><a class="nav-link" href="/about">About</a></li>
                <li class="nav-item"><a class="nav-link" href="/contact">Contact</a></li>
            </ul>
        </div>
    </div>
</nav>
""")
    print(f"Created navigation template at {nav_template}")

# Check if we have a footer template
footer_template = template_dir / "footer.html"
if not footer_template.exists():
    # Create a simple footer template
    with open(footer_template, "w") as footer_file:
        footer_file.write("""<!-- Footer Template -->
<footer class="footer mt-auto py-3 bg-light">
    <div class="container text-center">
        <span class="text-muted">&copy; 2023 ImpressionCore. All rights reserved.</span>
    </div>
</footer>
""")
    print(f"Created footer template at {footer_template}")

# Update base.html to include navigation and footer
nav_include_pattern = r'{%\s*include\s*[\'"]navigation\.html[\'"]\s*%}'
footer_include_pattern = r'{%\s*include\s*[\'"]footer\.html[\'"]\s*%}'

nav_include_str = '{% include "navigation.html" %}'
footer_include_str = '{% include "footer.html" %}'

# Update the base.html content
updated_base_content = base_content

# Add navigation if it doesn't exist
if not re.search(nav_include_pattern, updated_base_content):
    # Try to find where to insert the navigation
    updated_base_content = re.sub(
        r'<body>\s*', 
        f'<body>\n    {nav_include_str}\n    ', 
        updated_base_content
    )
    print("Added navigation include to base.html")

# Add footer if it doesn't exist
if not re.search(footer_include_pattern, updated_base_content):
    # Try to find where to insert the footer
    updated_base_content = re.sub(
        r'</div>\s*</body>', 
        f'</div>\n    \n    {footer_include_str}\n</body>', 
        updated_base_content
    )
    # If the pattern wasn't found, try an alternative insertion point
    if updated_base_content == base_content:
        updated_base_content = re.sub(
            r'</body>', 
            f'    {footer_include_str}\n</body>', 
            updated_base_content
        )
    print("Added footer include to base.html")

# Write the updated content back to the file
with open(base_html_path, "w") as file:
    file.write(updated_base_content)

print(f"Updated {base_html_path}")

## Update server.py

Finally, we'll modify server.py to ensure it correctly serves the templates listed in templates_listing.md.

In [ ]:
# Define path to server.py
server_py_path = Path("src/web/server.py")

# Check if server.py exists
if not server_py_path.exists():
    # Create directory if it doesn't exist
    server_py_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create a basic Flask server.py file
    with open(server_py_path, "w") as server_file:
        server_file.write("""from flask import Flask, render_template

app = Flask(__name__)

@app.route('/')
def home():
    return render_template('index.html')

# Additional routes will be added here

if __name__ == '__main__':
    app.run(debug=True)
""")
    print(f"Created basic {server_py_path}")

# Read the current server.py file
with open(server_py_path, "r") as file:
    server_content = file.read()

# Create a backup of the original file
backup_path = server_py_path.with_suffix(".py.bak")
shutil.copy2(server_py_path, backup_path)
print(f"Created backup of server.py at {backup_path}")

# Update the server.py to include routes for all templates
# We'll examine template files and create routes for them if they don't exist already

# Filter out templates that shouldn't have direct routes
template_routes = []
for file in template_files:
    filename = file.name
    # Skip base templates, includes, partials, components
    if (filename.startswith('base.') or 
        filename.startswith('_') or 
        'include' in filename.lower() or 
        'partial' in filename.lower() or 
        'component' in filename.lower() or
        'layout' in filename.lower() or
        'navigation' in filename.lower() or
        'footer' in filename.lower()):
        continue
    
    # Get route name from filename (remove .html extension)
    route_name = filename.replace('.html', '')
    
    # Special case for index/home
    if route_name == 'index' or route_name == 'home':
        route_path = '/'
    else:
        route_path = f'/{route_name}'
    
    template_routes.append((route_path, route_name, filename))

# Build the new routes as Python code
new_routes_code = ""

for route_path, route_name, filename in template_routes:
    # Check if the route already exists in the server.py file
    route_pattern = rf'@app\.route\([\'"]({re.escape(route_path)}|/{route_name})[\'"]'
    if not re.search(route_pattern, server_content):
        # Create a new route
        new_route = f"""
@app.route('{route_path}')
def {route_name.replace('-', '_')}():
    return render_template('{filename}')
"""
        new_routes_code += new_route

# Insert the new routes into server.py
if new_routes_code:
    # Look for where to add the new routes
    if '# Additional routes will be added here' in server_content:
        server_content = server_content.replace(
            '# Additional routes will be added here',
            '# Additional routes will be added here' + new_routes_code
        )
    else:
        # Try to insert before the if __name__ == '__main__': block
        main_pattern = r'if\s+__name__\s*==\s*[\'"]__main__[\'"]\s*:'
        match = re.search(main_pattern, server_content)
        if match:
            insert_position = match.start()
            server_content = (
                server_content[:insert_position] + 
                "\n# Generated routes\n" + new_routes_code + 
                "\n" + server_content[insert_position:]
            )
        else:
            # Just append at the end
            server_content += "\n# Generated routes\n" + new_routes_code

    # Write updated content back to server.py
    with open(server_py_path, "w") as file:
        file.write(server_content)
    
    print(f"Added {len(template_routes)} new routes to {server_py_path}")
else:
    print("No new routes needed to be added to server.py")

## Summary

This notebook has successfully:

1. Listed all template files in the 'src/web/templates' directory
2. Created a Markdown file (templates_listing.md) documenting these templates and their relationships
3. Updated base.html to include navigation and footer templates
4. Updated server.py to ensure it has routes for all relevant templates

Next steps could include:
- Regular maintenance of these files as new templates are added
- Adding more complex routing logic based on template content
- Creating additional template management utilities